# Baseline 2: Top-K recommendation by popularity

## 1. Load and clean the data

In [2]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

In [3]:
DATASET_DIRECTORY = os.path.join('..', 'data', 'ml-32m')

ratings = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'ratings.csv'))
movies = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'movies.csv'))
print(f'Ratings shape: {ratings.shape}')
print(f'Movies shape: {movies.shape}')

Ratings shape: (32000204, 4)
Movies shape: (87585, 3)


In [4]:
MIN_RATINGS_PER_USER = 10

ratings = ratings.dropna(subset=['userId', 'movieId', 'rating', 'timestamp'])
ratings = ratings[ratings.rating.between(0.5, 5.0)]
ratings = ratings.drop_duplicates(['userId', 'movieId'])

ratings_per_user = ratings.groupby('userId').movieId.transform('size')
ratings = ratings[ratings_per_user >= MIN_RATINGS_PER_USER].reset_index(drop=True)

## 2. Training split

In [5]:
from helpers.temporal_split import temporal_split

train, val, test = temporal_split(ratings)
print(len(train), len(val), len(test))

25605546 3197329 3197329


## 3. Scoring functions

- **Precision@K / Recall@K / NDCG@K:** how good the top-K recommendations are (higher is better).

In [6]:
def ndcg_at_k(recommended, relevant, k):
    credit = []
    for rank in range(k):
        credit.append(1.0 / np.log2(rank + 2))

    top_k = recommended[:k]
    score = 0.0
    for rank in range(len(top_k)):
        if top_k[rank] in relevant:
            score += credit[rank]

    hits_possible = min(len(relevant), k)
    best_possible = 0.0
    for rank in range(hits_possible):
        best_possible += credit[rank]

    if best_possible > 0:
        return score / best_possible
    return 0.0

def movies_by_user(df):
    result = {}
    for user_id, user_movies in df.groupby('userId').movieId:
        result[user_id] = set(user_movies.tolist())
    return result

def evaluate_ranking(recommend, history, held_out, cutoffs=(5,10), like_threshold=4.0):
    seen = movies_by_user(history)
    liked = held_out[held_out.rating >= like_threshold]
    relevant = movies_by_user(liked)

    scores = {}
    for k in cutoffs:
        scores[f'P@{k}'] = []
        scores[f'R@{k}'] = []
        scores[f'NDCG@{k}'] = []

    biggest_k = max(cutoffs)
    for user_id in relevant:
        correct = relevant[user_id]
        already_seen = seen.get(user_id, set())
        recommended = recommend(user_id, already_seen, biggest_k)

        for k in cutoffs:
            hits = 0
            for movie in recommended[:k]:
                if movie in correct:
                    hits += 1
            scores[f'P@{k}'].append(hits / k)
            scores[f'R@{k}'].append(hits / len(correct))
            scores[f'NDCG@{k}'].append(ndcg_at_k(recommended, correct, k))

    averages = {}
    for name in scores:
        averages[name] = float(np.mean(scores[name]))
    averages['n_users'] = len(relevant)
    return averages

## 4. Popularity ranking

In [7]:
counts = train.groupby('movieId').size().reset_index(name='n_ratings')
counts = counts.sort_values(['n_ratings', 'movieId'], ascending=[False, True])

popularity_order = counts.movieId.tolist()
count_of = dict(zip(counts.movieId, counts.n_ratings))
movie_title = dict(zip(movies.movieId, movies.title))

print('Most-rated movies in the training set:')
for movie in popularity_order[:10]:
    print(f'{count_of[movie]:>8}  {movie_title[movie]}')

Most-rated movies in the training set:
   95407  Shawshank Redemption, The (1994)
   90543  Forrest Gump (1994)
   89524  Pulp Fiction (1994)
   86373  Matrix, The (1999)
   82016  Silence of the Lambs, The (1991)
   77399  Star Wars: Episode IV - A New Hope (1977)
   69866  Fight Club (1999)
   66949  Jurassic Park (1993)
   66416  Schindler's List (1993)
   66381  Lord of the Rings: The Fellowship of the Ring, The (2001)


## 5. Recomming unseen

In [8]:
def recommend_popular(user_id, seen, n):
    recommendations = []
    for movie in popularity_order:
        if movie not in seen:
            recommendations.append(movie)
        if len(recommendations) == n:
            break
    return recommendations

## 6. results

In [9]:
val_results = evaluate_ranking(recommend_popular, train, val)

history = pd.concat([train, val])
test_results = evaluate_ranking(recommend_popular, history, test)

ranking_results = pd.DataFrame([val_results, test_results], index=['Validation', 'Test'])
display(ranking_results.round(4))

,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10,n_users
Validation,0.0362,0.0328,0.0447,0.0312,0.0555,0.0502,188220
Test,0.0355,0.0306,0.0436,0.0300,0.0503,0.0476,188901


## 7. Popularity + favorite genres

In [ ]:
LIKE_THRESHOLD = 3.5
TOP_GENRES = 3

genres_of = {}
for movie_id, genre_text in zip(movies.movieId, movies.genres):
    if pd.isna(genre_text) or genre_text == '(no genres listed)':
        genres_of[movie_id] = set()
    else:
        genres_of[movie_id] = set(genre_text.split('|'))

favorite_genres = {}
liked_train = train[train.rating >= LIKE_THRESHOLD]
for user_id, user_movies in liked_train.groupby('userId').movieId:
    genre_counts = {}
    for movie in user_movies:
        for genre in genres_of.get(movie, set()):
            genre_counts[genre] = genre_counts.get(genre, 0) + 1

    pairs = []
    for genre in genre_counts:
        pairs.append((genre_counts[genre], genre))
    pairs.sort(reverse=True)

    top_genres = set()
    for count, genre in pairs[:TOP_GENRES]:
        top_genres.add(genre)
    favorite_genres[user_id] = top_genres

Found favorite genres for 200740 users


In [ ]:
def recommend_popular_in_favorite_genres(user_id, seen, n):
    wanted = favorite_genres.get(user_id, set())
    if len(wanted) == 0:
        return recommend_popular(user_id, seen, n) # if they don't watch anything then just give them the most popular

    recommendations = []
    for movie in popularity_order:
        if movie in seen:
            continue
        movie_genres = genres_of.get(movie, set())
        if len(movie_genres & wanted) > 0:
            recommendations.append(movie)
        if len(recommendations) == n:
            break
    return recommendations

## 8. Movies like a given movie

In [28]:
MIN_RATINGS_FOR_SIMILAR = 50

def show_movies_like(title_text, n=10):
    matches = movies[movies.title.str.contains(title_text, case=False, regex=False)]
    for movie_id in matches.movieId:
        query_genres = genres_of.get(movie_id, set())
        if len(query_genres) == 0:
            continue

        candidates = []
        for other_id in popularity_order:
            if count_of[other_id] < MIN_RATINGS_FOR_SIMILAR:
                break
            if other_id == movie_id:
                continue
            other_genres = genres_of.get(other_id, set())
            shared = query_genres & other_genres
            if len(shared) == 0:
                continue
            overlap = len(shared) / len(query_genres | other_genres)
            candidates.append((overlap, count_of[other_id], other_id))

        candidates.sort(reverse=True)

        genre_text = ', '.join(sorted(query_genres))
        print(f'Query: {movie_title[movie_id]} [{genre_text}]')
        for overlap, n_ratings, other_id in candidates[:n]:
            print(f'{overlap:.2f}  {n_ratings:>7} ratings  {movie_title[other_id]}')
        return
    print('No movie with genres found for', title_text)

In [29]:
show_movies_like('Saw (2003)')

Query: Saw (2003) [Crime, Horror]
1.00    22816 ratings  Psycho (1960)
1.00      153 ratings  Henry: Portrait of a Serial Killer, Part 2 (1998)
1.00       52 ratings  King of the Ants (2003)
0.67    82016 ratings  Silence of the Lambs, The (1991)
0.67     4559 ratings  Serial Mom (1994)
0.67     2720 ratings  Saw III (2006)
0.67     2570 ratings  Psycho (1998)
0.67     2211 ratings  Purge, The (2013)
0.67     1851 ratings  Saw IV (2007)
0.67     1833 ratings  Cabinet of Dr. Caligari, The (Cabinet des Dr. Caligari., Das) (1920)


## 9. Results with genres

In [26]:
genre_val_results = evaluate_ranking(recommend_popular_in_favorite_genres, train, val)
genre_test_results = evaluate_ranking(recommend_popular_in_favorite_genres, history, test)
genre_results = pd.DataFrame([genre_val_results, genre_test_results], index=['Validation', 'Test'])

In [27]:
print("popularity")
display(ranking_results.round(4))

print()
print("popularity + genre")
display(genre_results)

popularity


,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10,n_users
Validation,0.0362,0.0328,0.0447,0.0312,0.0555,0.0502,188220
Test,0.0355,0.0306,0.0436,0.0300,0.0503,0.0476,188901



popularity + genre


,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10,n_users
Validation,0.036183,0.033852,0.044853,0.031216,0.057254,0.050758,188220
Test,0.035200,0.031049,0.043379,0.029694,0.051034,0.047510,188901
